# ML-07 — Baseline Action Score and Top-10 Review

This notebook turns the refresh lane into a transparent rule: first prove two signals are real, then encode one readable score with reason codes, then read the ranked queue with a skeptic's eye. It also writes the queue CSV to `work/outputs/baseline_action_score.csv` so the later model has a baseline to beat.

## 1. My rule and its reason codes

I am using a refresh-review rule: pages rise when they are visible, stale, or showing CTR / position decay. One of the checked signals is staleness, which backs FlyRank's refresh flags; the other is CTR-vs-position, which backs the CTR-fix logic. Reason codes: `stale_visible_page`, `low_ctr_visible_page`, `page_one_decay_risk`, `thin_visible_page`, `low_engagement_visible_page`, and the fallback `general_refresh_review`.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

repo_root = next(
    (candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (candidate / "scripts" / "ml_utils.py").exists()),
    None,
 )
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

sys.path.insert(0, str(repo_root / "scripts"))
from ml_utils import normalize, percentile_rank, precision_at_k, write_json, OUTPUT_DIR

input_path = repo_root / "data" / "processed" / "refresh_feature_vector.csv"
queue_input = pd.read_csv(input_path)

def bucket_table(frame: pd.DataFrame, source_column: str, bins: list[float], labels: list[str], table_name: str) -> pd.DataFrame:
    work = frame.copy()
    work[table_name] = pd.cut(work[source_column], bins=bins, labels=labels, include_lowest=True)
    table = (
        work.groupby(table_name, dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(table_name, kind="stable")
        .reset_index(drop=True)
    )
    display(table)
    return table

staleness_bins = [-0.1, 29, 89, 179, 364, np.inf]
staleness_labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
staleness_table = bucket_table(
    queue_input,
    "days_since_last_update",
    staleness_bins,
    staleness_labels,
    "staleness_band",
)
stale_visible_n = int(
    ((queue_input["days_since_last_update"] >= 180) & (queue_input["impressions_90d"] >= 500)).sum()
 )
stale_verdict = "CONFIRMED" if stale_visible_n >= 100 else ("MIXED" if stale_visible_n > 0 else "FALSE")
print(f"Staleness signal verdict: {stale_verdict}")
print(f"stale_visible_page n = {stale_visible_n}")

ctr_position_frame = queue_input.loc[queue_input["impressions_90d"] >= 500].copy()
ctr_position_frame["position_band"] = pd.cut(
    ctr_position_frame["avg_position"],
    bins=[-0.1, 0, 3, 10, 20, np.inf],
    labels=["0", "1-3", "4-10", "11-20", "20+"],
    include_lowest=True,
 )
ctr_position_frame["ctr_band"] = pd.cut(
    ctr_position_frame["ctr"],
    bins=[-0.1, 0.1, 0.5, 1.0, 2.0, np.inf],
    labels=["0-0.1", "0.1-0.5", "0.5-1.0", "1.0-2.0", "2.0+"],
    include_lowest=True,
 )
ctr_table = (
    ctr_position_frame.groupby(["position_band", "ctr_band"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["position_band", "ctr_band"], kind="stable")
    .reset_index(drop=True)
 )
display(ctr_table)

low_ctr_visible_n = int(
    (
        (queue_input["impressions_90d"] >= 500)
        & (queue_input["avg_position"].between(1, 20, inclusive="both"))
        & (queue_input["ctr"] < 0.5)
    ).sum()
 )
ctr_verdict = "CONFIRMED" if low_ctr_visible_n >= 100 else ("MIXED" if low_ctr_visible_n > 0 else "FALSE")
print(f"CTR-vs-position signal verdict: {ctr_verdict}")
print(f"low_ctr_visible_page n = {low_ctr_visible_n}")

def reason_codes(row: pd.Series) -> list[str]:
    reasons: list[str] = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return reasons

def suggested_action(row: pd.Series) -> str:
    reasons = set(str(row["reason_codes"]).split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons:
        return "refresh"
    return "monitor"

print("Rule in plain words: review pages that are stale, visible, thin, or low-CTR before the rest.")
print("This rule is deliberately transparent so a person can read why each row scored well.")

,staleness_band,n
0,0-30,20480
1,31-90,175
2,91-180,9171
3,181-365,169
4,365+,5


Staleness signal verdict: MIXED
stale_visible_page n = 17


,position_band,ctr_band,n
0,1-3,0-0.1,154
1,1-3,0.1-0.5,212
2,1-3,0.5-1.0,86
3,1-3,1.0-2.0,26
4,1-3,2.0+,2
5,4-10,0-0.1,1528
6,4-10,0.1-0.5,4119
7,4-10,0.5-1.0,1098
8,4-10,1.0-2.0,306
9,4-10,2.0+,33


CTR-vs-position signal verdict: CONFIRMED
low_ctr_visible_page n = 9745
Rule in plain words: review pages that are stale, visible, thin, or low-CTR before the rest.
This rule is deliberately transparent so a person can read why each row scored well.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
baseline_frame = queue_input.copy()

baseline_frame["visibility_score"] = percentile_rank(np.log1p(baseline_frame["impressions_90d"]))
baseline_frame["freshness_risk_score"] = percentile_rank(baseline_frame["days_since_last_update"])
baseline_frame["position_opportunity_score"] = (
    (1 - normalize(baseline_frame["avg_position"].clip(lower=1, upper=50)))
    * baseline_frame["visibility_score"]
    * (baseline_frame["avg_position"] > 0).astype(int)
)
baseline_frame["depth_gap_score"] = (1 - percentile_rank(baseline_frame["word_count"])) * baseline_frame["visibility_score"]

baseline_frame["baseline_refresh_score"] = (
    0.40 * baseline_frame["visibility_score"]
    + 0.30 * baseline_frame["freshness_risk_score"]
    + 0.25 * baseline_frame["position_opportunity_score"]
    + 0.05 * baseline_frame["depth_gap_score"]
).clip(0, 1)

baseline_frame["reason_codes"] = baseline_frame.apply(lambda row: "|".join(reason_codes(row)), axis=1)
baseline_frame["suggested_action_baseline"] = baseline_frame.apply(suggested_action, axis=1)
baseline_frame["baseline_rank"] = baseline_frame["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action_baseline",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
]

queue_output = baseline_frame[output_columns].sort_values("baseline_rank").reset_index(drop=True)

work_outputs = repo_root / "work" / "outputs"
work_outputs.mkdir(parents=True, exist_ok=True)
queue_csv_path = work_outputs / "baseline_action_score.csv"
queue_output.to_csv(queue_csv_path, index=False)

write_json(
    work_outputs / "baseline_metadata.json",
    {
        "rows": int(len(queue_output)),
        "top_score": float(queue_output["baseline_refresh_score"].max()),
        "median_score": float(queue_output["baseline_refresh_score"].median()),
        "declining_rate_top_50": float(queue_output.head(50)["is_declining_label"].mean()) if len(queue_output) else 0.0,
        "score_formula": {
            "visibility_score": 0.40,
            "freshness_risk_score": 0.30,
            "position_opportunity_score": 0.25,
            "depth_gap_score": 0.05,
        },
    },
)

base_rate = float(queue_output["is_declining_label"].mean())
precision_at_10 = precision_at_k(queue_output["is_declining_label"], queue_output["baseline_refresh_score"], 10)

print(f"Wrote queue CSV: {queue_csv_path}")
print(f"Base rate: {base_rate:.3f}")
print(f"precision@10: {precision_at_10:.3f}")
display(queue_output.head(10))

Wrote queue CSV: c:\Users\kunal\Documents\Project\Internship\FlyRank\Google-Search-Ranking-Discoverability\work\outputs\baseline_action_score.csv
Base rate: 0.542
precision@10: 0.200


,content_id,client_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action_baseline,...,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction
0,content_9532f197bbc8,client_4e07408562,1,0.941189,0.999633,0.8432,0.979233,0.871347,page_one_decay_risk|low_engagement_visible_page,monitor,...,2689,1098,2.0,0.87,8.01,28.75,445,104,0.0,down
1,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,0.994167,0.8432,0.963733,0.866582,page_one_decay_risk|low_engagement_visible_page,monitor,...,512,549,2.5,0.52,7.47,13.15,329,104,0.0,stable
2,content_07f2e7a6f38a,client_19581e27de,3,0.934080,0.994467,0.8432,0.959965,0.866843,page_one_decay_risk|low_engagement_visible_page,monitor,...,856,780,2.7,0.85,2.05,4.60,313,104,0.0,stable
3,content_e5ae436f9a16,client_4e07408562,4,0.933606,0.996000,0.8432,0.955347,0.868180,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,533,522,3.0,0.45,7.09,12.60,421,104,0.0,stable
4,content_3430a8b94511,client_19581e27de,5,0.933559,0.998167,0.8432,0.951314,0.870069,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,440,534,3.3,0.29,6.18,11.04,329,104,0.0,stable
5,content_cbd93118300b,client_19581e27de,6,0.933263,0.997733,0.8432,0.950901,0.869691,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,662,535,3.3,0.46,1.87,5.38,313,104,0.0,down
6,content_9c195417f6ef,client_19581e27de,7,0.932991,0.991400,0.8432,0.961051,0.864170,page_one_decay_risk|low_engagement_visible_page,monitor,...,574,515,2.5,0.73,1.55,2.79,313,104,0.0,stable
7,content_ba2acb4ebd04,client_19581e27de,8,0.931623,0.997567,0.8432,0.944635,0.869546,page_one_decay_risk|low_engagement_visible_page,monitor,...,1185,1147,3.6,0.83,1.92,5.08,362,104,0.0,stable
8,content_79b25654070a,client_19581e27de,9,0.931363,0.997933,0.8432,0.942945,0.869865,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,711,619,3.7,0.48,2.26,3.46,257,104,0.0,stable
9,content_adddad39251c,client_19581e27de,10,0.931124,0.996833,0.8432,0.943940,0.868906,page_one_decay_risk|low_engagement_visible_page,monitor,...,711,688,3.6,0.55,3.92,6.32,329,104,0.0,stable


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [4]:
def review_why(row: pd.Series) -> str:
    reasons = str(row["reason_codes"]).split("|")
    parts: list[str] = []
    if "stale_visible_page" in reasons:
        parts.append("it is old and still has enough demand to justify a refresh")
    if "low_ctr_visible_page" in reasons:
        parts.append("it is visible, but the CTR is low enough to justify a title / snippet review")
    if "page_one_decay_risk" in reasons:
        parts.append("it is already near page-one territory, so small changes may matter")
    if "thin_visible_page" in reasons:
        parts.append("the page looks thin relative to its visibility")
    if "low_engagement_visible_page" in reasons:
        parts.append("the page is getting sessions but not enough engagement")
    if not parts:
        parts.append("it has enough visibility to deserve a manual look")
    return "; ".join(parts)

def wrong_if(row: pd.Series) -> str:
    reasons = set(str(row["reason_codes"]).split("|"))
    checks: list[str] = []
    if "stale_visible_page" in reasons:
        checks.append("it was updated recently or the age signal is not real")
    if "low_ctr_visible_page" in reasons:
        checks.append("the page is actually too low in position for a CTR fix to help")
    if "page_one_decay_risk" in reasons:
        checks.append("it is not really sitting in the page-one range")
    if "thin_visible_page" in reasons:
        checks.append("the short length is intentional or the word count is missing")
    if "low_engagement_visible_page" in reasons:
        checks.append("the engagement signal is noisy or caused by a short-lived traffic swing")
    if not checks:
        checks.append("there is no specific underlying signal, so it is only a weak general review candidate")
    return "; ".join(checks)

top10_review = queue_output.head(10).copy()
top10_review["why_it_is_here"] = top10_review.apply(review_why, axis=1)
top10_review["what_would_make_it_wrong"] = top10_review.apply(wrong_if, axis=1)
top10_review["confidence_note"] = top10_review["reason_codes"].str.split("|").str.len().map(lambda n: f"{n} reason code(s)")

display(
    top10_review[[
        "baseline_rank",
        "content_id",
        "client_id",
        "suggested_action_baseline",
        "reason_codes",
        "confidence_note",
        "why_it_is_here",
        "what_would_make_it_wrong",
    ]]
 )

,baseline_rank,content_id,client_id,suggested_action_baseline,reason_codes,confidence_note,why_it_is_here,what_would_make_it_wrong
0,1,content_9532f197bbc8,client_4e07408562,monitor,page_one_decay_risk|low_engagement_visible_page,2 reason code(s),"it is already near page-one territory, so smal...",it is not really sitting in the page-one range...
1,2,content_4d1fe5b32dc2,client_19581e27de,monitor,page_one_decay_risk|low_engagement_visible_page,2 reason code(s),"it is already near page-one territory, so smal...",it is not really sitting in the page-one range...
2,3,content_07f2e7a6f38a,client_19581e27de,monitor,page_one_decay_risk|low_engagement_visible_page,2 reason code(s),"it is already near page-one territory, so smal...",it is not really sitting in the page-one range...
3,4,content_e5ae436f9a16,client_4e07408562,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,3 reason code(s),"it is visible, but the CTR is low enough to ju...",the page is actually too low in position for a...
4,5,content_3430a8b94511,client_19581e27de,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,3 reason code(s),"it is visible, but the CTR is low enough to ju...",the page is actually too low in position for a...
5,6,content_cbd93118300b,client_19581e27de,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,3 reason code(s),"it is visible, but the CTR is low enough to ju...",the page is actually too low in position for a...
6,7,content_9c195417f6ef,client_19581e27de,monitor,page_one_decay_risk|low_engagement_visible_page,2 reason code(s),"it is already near page-one territory, so smal...",it is not really sitting in the page-one range...
7,8,content_ba2acb4ebd04,client_19581e27de,monitor,page_one_decay_risk|low_engagement_visible_page,2 reason code(s),"it is already near page-one territory, so smal...",it is not really sitting in the page-one range...
8,9,content_79b25654070a,client_19581e27de,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,3 reason code(s),"it is visible, but the CTR is low enough to ju...",the page is actually too low in position for a...
9,10,content_adddad39251c,client_19581e27de,monitor,page_one_decay_risk|low_engagement_visible_page,2 reason code(s),"it is already near page-one territory, so smal...",it is not really sitting in the page-one range...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
queue_output["reason_count"] = queue_output["reason_codes"].str.split("|").str.len()
weak_picks = queue_output.loc[queue_output["reason_count"] <= 1].head(5).copy()

print("Weak picks are the rows with only one reason code or the generic fallback.")
display(weak_picks[["baseline_rank", "content_id", "baseline_refresh_score", "reason_codes", "suggested_action_baseline"]])

score_inputs_used = {"impressions_90d", "days_since_last_update", "avg_position", "word_count"}
blocked_inputs = {"is_declining_label", "trend_direction", "trend_pct", "april_impressions"}

leakage_violations = sorted(score_inputs_used.intersection(blocked_inputs))
leakage_check = pd.DataFrame(
    [
        {
            "check": "No label-derived inputs in the score",
            "result": "PASS" if not leakage_violations else "FAIL",
            "detail": "The score inputs are only impressions, freshness, position, and depth; no label columns are used.",
        },
        {
            "check": "No future-window columns in the score",
            "result": "PASS" if "april_impressions" not in score_inputs_used else "FAIL",
            "detail": "The score does not read the April outcome window.",
        },
        {
            "check": "No product flags used as features",
            "result": "PASS",
            "detail": "The baseline uses observable signals only; no hidden FlyRank flag columns are needed.",
        },
    ]
)

display(leakage_check)

print("Excluded from the features on purpose: trend_direction, trend_pct, and is_declining_label.")
print("The notebook keeps those fields for evaluation and explanation only, not for scoring.")

Weak picks are the rows with only one reason code or the generic fallback.


,baseline_rank,content_id,baseline_refresh_score,reason_codes,suggested_action_baseline
60,61,content_69fad7e6c50c,0.920633,low_engagement_visible_page,monitor
92,93,content_03d2673b2553,0.916304,low_engagement_visible_page,monitor
145,146,content_654d006adc44,0.909312,low_engagement_visible_page,monitor
146,147,content_c1143eda3230,0.909260,low_engagement_visible_page,monitor
158,159,content_5184b85dc6dd,0.908353,low_engagement_visible_page,monitor


,check,result,detail
0,No label-derived inputs in the score,PASS,"The score inputs are only impressions, freshne..."
1,No future-window columns in the score,PASS,The score does not read the April outcome window.
2,No product flags used as features,PASS,The baseline uses observable signals only; no ...


Excluded from the features on purpose: trend_direction, trend_pct, and is_declining_label.
The notebook keeps those fields for evaluation and explanation only, not for scoring.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.